# CoinCell — Model Training (Kaggle CPU)

**Congressional App Challenge** · Trains DualViewNet + CoinCellNet on CPU

Before running:
1. **Settings → Internet** ON (GPU not required)
2. **Optional:** Add-ons → Secrets → `HF_TOKEN` for auto-upload to Hugging Face Hub

Only uses `/kaggle/working/` — does not touch your other Kaggle notebooks/datasets.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path

WORK = Path('/kaggle/working')
SRC = WORK / 'coincell-src'
REPO = 'https://github.com/arjunkshah12345-hash/coincell.git'

if not SRC.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO, str(SRC)], check=True)
else:
    subprocess.run(['git', '-C', str(SRC), 'pull', '--ff-only'], check=False)

sys.path.insert(0, str(SRC))
print('Source:', SRC)
print('GPU available:', __import__('torch').cuda.is_available())

In [ ]:
import torch
from coincell.classifier import train_models, save_models

DEVICE = 'cpu'  # CPU training — no GPU needed
print(f'Training on {DEVICE}')

from coincell import classifier
orig_build = classifier.build_dataset

def cpu_build(n_per_class=150, size=224, dual=True):
    return orig_build(n_per_class=n_per_class, size=size, dual=dual)

classifier.build_dataset = cpu_build

EPOCHS = 8
BATCH = 32
single, dual = train_models(epochs=EPOCHS, batch_size=BATCH, device=DEVICE)

weights_path = WORK / 'coincell.pt'
save_models(single, dual, weights_path)
print(f'Saved → {weights_path} ({weights_path.stat().st_size / 1024:.0f} KB)')

In [ ]:
# Evaluate ensemble (CV + Kaggle-trained CNN)
import os
os.environ['COINCELL_WEIGHTS'] = str(WORK / 'coincell.pt')

# Clear cached engine so it reloads Kaggle weights
import coincell.inference as inf
inf._engine = None

metrics = evaluate_on_synthetic(n=80)
metrics_path = WORK / 'metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

In [ ]:
# Upload to Hugging Face Hub (optional — needs HF_TOKEN secret)
try:
    from huggingface_hub import HfApi, create_repo
    from kaggle_secrets import UserSecretsClient
    HF_REPO = 'arjunkshah12345-hash/coincell-weights'
    token = UserSecretsClient().get_secret('HF_TOKEN')
    api = HfApi(token=token)
    create_repo(HF_REPO, repo_type='model', exist_ok=True)
    api.upload_file(str(weights_path), 'coincell.pt', repo_id=HF_REPO, repo_type='model',
                    commit_message=f'CoinCell Kaggle CPU train — {EPOCHS} epochs')
    api.upload_file(str(metrics_path), 'metrics.json', repo_id=HF_REPO, repo_type='model',
                    commit_message='Evaluation metrics')
    print(f'✓ Weights live: https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'HF upload skipped ({e}). Weights saved to Kaggle output: coincell.pt')